In [1]:
import pandas as pd
df = pd.read_csv("airports.csv", engine='python', on_bad_lines='skip')

In [2]:
df

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,NaN
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NaN,US,US-AL,Harvest,no,NaN,NaN,00AL,00AL,NaN,NaN,NaN
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80.0,NaN,US,US-AK,King Salmon,no,NaN,NaN,00AN,00AN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85340,32753,ZYYY,medium_airport,Shenyang Dongta Airport,41.784354,123.496308,157.0,AS,CN,CN-21,"Dadong, Shenyang",no,ZYYY,NaN,ZYYY,NaN,NaN,NaN,"东塔机场, SHE"
85341,46378,ZZ-0001,heliport,Sealand Helipad,51.894444,1.482500,40.0,EU,GB,GB-ENG,Sealand,no,NaN,NaN,NaN,NaN,http://www.sealandgov.org/,https://en.wikipedia.org/wiki/Principality_of_...,Roughs Tower Helipad
85342,307326,ZZ-0002,small_airport,Glorioso Islands Airstrip,-11.584278,47.296389,11.0,AF,TF,TF-U-A,Grande Glorieuse,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN
85343,346788,ZZ-0003,small_airport,Fainting Goat Airport,32.110587,-97.356312,690.0,NaN,US,US-TX,Blum,no,NaN,NaN,87TX,87TX,NaN,NaN,NaN


In [3]:
df.loc[df['gps_code'].str.startswith('K', na=False), 'ident'].unique()

array(['00A', '00N', '00S', ..., 'Y77', 'Y87', 'Y88'],
      shape=(3623,), dtype=object)

In [4]:
df[df['gps_code']=="K00M"]

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
37475,18304,K00M,small_airport,Thigpen Field,31.953728,-89.23527,351.0,NaN,US,US-MS,Bay Springs,no,NaN,NaN,K00M,00M,NaN,NaN,NaN


In [5]:
# 1. Filter for US airports and ensure we only have rows with an ICAO code
geo_df = df[df['ident'].str.startswith('K', na=False)].copy()
# geo_df = df[df['ident'].str.startswith('K', na=False)].copy()
# geo_df = df[df['gps_code'].str.startswith('K', na=False)].copy()


# 2. Select the specific columns requested
geo_df = geo_df[['ident' ,'latitude_deg', 'longitude_deg']]

# 3. Drop rows with missing coordinates or missing ICAO codes
geo_df = geo_df.dropna(subset=['ident', 'latitude_deg', 'longitude_deg'])

# 4. Remove duplicates to ensure each ICAO code appears only once
geo_df = geo_df.drop_duplicates(subset=['ident'])

# Display result
print(f"Total unique ICAO airports: {len(geo_df)}")
geo_df.head()

Total unique ICAO airports: 5519


,ident,latitude_deg,longitude_deg
37473,K00C,37.203201,-107.869003
37474,K00F,45.470462,-105.457145
37475,K00M,31.953728,-89.235270
37476,K00R,30.685900,-95.017899
37477,K00V,38.945801,-104.570000


In [6]:
geo_df.to_csv("icao_geo_location.csv")

In [7]:
from timezonefinder import TimezoneFinder
import pytz

tf = TimezoneFinder()

def get_timezone_info(row):
    # Step 1: Get IANA timezone from coordinates
    # Using the column names 'latitude_deg' and 'longitude_deg' from your geo_df
    tz_name = tf.timezone_at(lat=row['latitude_deg'], lng=row['longitude_deg'])
    
    # Return 'Unknown' if coordinates are outside a defined timezone
    if tz_name is None:
        return pd.Series([None, None, None, None])
    
    # Step 2 & 3: Time conversion (Assuming you have a 'flight_utc' variable or column)
    # If you have a specific UTC time to convert, use it here. 
    # For this example, I'll just return the timezone name.
    return pd.Series([tz_name])

# Apply to your dataframe
# This creates a new column called 'timezone'
geo_df[['timezone']] = geo_df.apply(get_timezone_info, axis=1)

geo_df.head()

,ident,latitude_deg,longitude_deg,timezone
37473,K00C,37.203201,-107.869003,America/Denver
37474,K00F,45.470462,-105.457145,America/Denver
37475,K00M,31.953728,-89.235270,America/Chicago
37476,K00R,30.685900,-95.017899,America/Chicago
37477,K00V,38.945801,-104.570000,America/Denver


In [8]:
geo_df.to_csv("icao_timezone.csv")

In [9]:
# 1. Create a copy and define a unified identifier
# We use 'ident' first, then fill missing values with 'gps_code'
df_unified = df.copy()
df_unified['unified_id'] = df_unified['ident'].fillna(df_unified['gps_code'])

# 2. Filter for US airports starting with 'K'
# This checks both the ident and gps_code logic
mask = (df_unified['ident'].str.startswith('K', na=False)) | \
       (df_unified['gps_code'].str.startswith('K', na=False))

geo_df_expanded = df_unified[mask].copy()

# 3. Select and clean the columns
# We keep both for reference, but use unified_id as the primary key
geo_df_expanded = geo_df_expanded[['unified_id', 'ident', 'gps_code', 'latitude_deg', 'longitude_deg']]

# 4. Drop rows missing essential mapping data
geo_df_expanded = geo_df_expanded.dropna(subset=['unified_id', 'latitude_deg', 'longitude_deg'])

# 5. Remove duplicates based on the unified ID
geo_df_expanded = geo_df_expanded.drop_duplicates(subset=['unified_id'])

print(f"Total unique airports found: {len(geo_df_expanded)}")
geo_df_expanded.head()

Total unique airports found: 5870


,unified_id,ident,gps_code,latitude_deg,longitude_deg
0,00A,00A,K00A,40.070985,-74.933689
30,00N,00N,K00N,39.472998,-75.184346
41,00S,00S,K00S,44.181466,-122.086000
49,00W,00W,K00W,46.672884,-117.441933
56,01A,01A,K01A,62.940833,-152.269722


In [10]:
geo_df_expanded.to_csv("icao_geo_location_expanded.csv")

In [11]:
from timezonefinder import TimezoneFinder
import pytz

tf = TimezoneFinder()

def get_timezone_info(row):
    # Step 1: Get IANA timezone from coordinates
    # Using the column names 'latitude_deg' and 'longitude_deg' from your geo_df
    tz_name = tf.timezone_at(lat=row['latitude_deg'], lng=row['longitude_deg'])
    
    # Return 'Unknown' if coordinates are outside a defined timezone
    if tz_name is None:
        return pd.Series([None, None, None, None])
    
    # Step 2 & 3: Time conversion (Assuming you have a 'flight_utc' variable or column)
    # If you have a specific UTC time to convert, use it here. 
    # For this example, I'll just return the timezone name.
    return pd.Series([tz_name])

# Apply to your dataframe
# This creates a new column called 'timezone'
geo_df_expanded[['timezone']] = geo_df_expanded.apply(get_timezone_info, axis=1)

geo_df_expanded.head()

,unified_id,ident,gps_code,latitude_deg,longitude_deg,timezone
0,00A,00A,K00A,40.070985,-74.933689,America/New_York
30,00N,00N,K00N,39.472998,-75.184346,America/New_York
41,00S,00S,K00S,44.181466,-122.086000,America/Los_Angeles
49,00W,00W,K00W,46.672884,-117.441933,America/Los_Angeles
56,01A,01A,K01A,62.940833,-152.269722,America/Anchorage


In [12]:
geo_df_expanded.to_csv("icao_timezone_expanded.csv")